# NCKH — Team Standard Training V2 — BERT — G1

**Scenario:** `G1` — Generic WordNet EDA two-stage + Minority-F2

This notebook is intentionally **single-seed** for the controlled ablation matrix.

- Shared ablation seed: `42`
- Same frozen train / validation / test files for every member
- Same BERT, ASL, LR, epochs and threshold protocol
- Do not regenerate augmentation files
- Do not change the seed for this ablation run



In [ ]:
# Kaggle bootstrap. P100 (Pascal/sm_60) needs a PyTorch wheel that still contains its CUDA kernels.
import subprocess, sys
gpu_name=subprocess.check_output(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],text=True).strip()
print("Detected GPU:",gpu_name)
if "P100" in gpu_name:
    print("[SETUP] Replacing only torch with the P100-compatible CUDA 12.1 wheel (no dependency upgrades)...")
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--force-reinstall","--no-deps","torch==2.5.1","--index-url","https://download.pytorch.org/whl/cu121"])
    subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchvision","torchaudio"],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.check_call([sys.executable,"-m","pip","install","-q","transformers>=4.48,<5","iterative-stratification","sentencepiece","tqdm"])
print("[OK] Kaggle dependencies installed")

In [ ]:

from pathlib import Path
import os, re, gc, json, math, time, random, pickle, platform, subprocess, warnings
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, hamming_loss
from sklearn.preprocessing import MultiLabelBinarizer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoConfig, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# EXPERIMENT SETUP
# ============================================================
SCENARIO = "G1"

# LOCKED seed for controlled ablation.
SEED = 42

# Do not change these per member / per scenario.
CONFIG = {
    "seed": SEED,
    "model_checkpoint": "google-bert/bert-base-uncased",
    "max_length": 384,
    "model_epochs": 6,
    "stage2_epochs": 6,
    "train_batch_size": 8,
    "eval_batch_size": 16,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "max_grad_norm": 1.0,
    "gamma_neg": 4.0,
    "gamma_pos": 1.0,
    "asl_clip": 0.05,
    "asl_eps": 1e-8,
    "threshold_min": 0.05,
    "threshold_max": 0.95,
    "threshold_step": 0.01,
    "min_val_support_per_label": 5,
    "num_workers": 2,
    "log_every_batches": 250,
    "early_stopping_patience": 2,
    "minority_support_threshold": 20,
    "minority_fbeta_beta": 2.0,
    "run_test": True,
    "force_rerun": False,
}

# ============================================================
# EXPERIMENT REGISTRY â€” DO NOT EDIT DURING PAPER RUNS
# ============================================================
EXPERIMENTS = {
    "A0": {
        "title": "Original-only one-stage baseline",
        "stage1_mode": "original",
        "stage2": False,
        "minority_f2": False,
    },
    "G0": {
        "title": "Generic WordNet EDA one-stage control",
        "stage1_mode": "generic",
        "stage2": False,
        "minority_f2": False,
    },
    "B1": {
        "title": "Cyber EDA one-stage",
        "stage1_mode": "cyber",
        "stage2": False,
        "minority_f2": False,
    },
    "G1": {
        "title": "Generic WordNet EDA two-stage + Minority-F2",
        "stage1_mode": "generic",
        "stage2": True,
        "minority_f2": True,
    },
    "B2_E1": {
        "title": "Cyber EDA two-stage; outputs B2 global and E1 Minority-F2",
        "stage1_mode": "cyber",
        "stage2": True,
        "minority_f2": True,
    },
}

if SCENARIO not in EXPERIMENTS:
    raise ValueError(f"Unknown SCENARIO={SCENARIO}. Valid: {sorted(EXPERIMENTS)}")

EXP = EXPERIMENTS[SCENARIO]

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path.cwd()
RESULTS = WORK_ROOT / "results" / "BERT" / SCENARIO
RESULTS.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("GPU is not enabled.")

gpu_capability = torch.cuda.get_device_capability(0)
compiled_arches = torch.cuda.get_arch_list()
required_arch = f"sm_{gpu_capability[0]}{gpu_capability[1]}"
if compiled_arches and required_arch not in compiled_arches:
    raise RuntimeError(
        f"Installed PyTorch lacks {required_arch}. Restart session and run bootstrap cell first."
    )

_cuda_smoke = torch.arange(8, device=DEVICE)
torch.cuda.synchronize()
del _cuda_smoke

print("=" * 78)
print("NCKH BERT TRAINING")
print("Scenario:", SCENARIO, "-", EXP["title"])
print("Seed:", CONFIG["seed"])
print("Device:", DEVICE, torch.cuda.get_device_name(0))
print("Results:", RESULTS)
print("=" * 78)
print(json.dumps(CONFIG, indent=2))


In [ ]:

# ============================================================
# DATASET SUBSET CONFIGURATION
# ============================================================
# Options: "joint" (default), "cti_to_mitre", "tram"
DATASET_SUBSET = "joint"


def find_one(filename, required=True, subset=DATASET_SUBSET):
    all_matches = sorted(INPUT_ROOT.rglob(filename))
    if not all_matches and not IS_KAGGLE:
        all_matches = sorted(Path.cwd().rglob(filename))
    if not all_matches:
        if required:
            raise FileNotFoundError(
                f"Cannot find {filename}. Put the dataset in Kaggle Input or workspace."
            )
        return None
    # Filter by subset directory if specified (e.g. joint, cti_to_mitre, tram)
    if subset:
        subset_matches = [
            p for p in all_matches if subset.lower() in [part.lower() for part in p.parts]
        ]
        if not subset_matches:
            subset_matches = [p for p in all_matches if subset.lower() in str(p).lower()]
        if subset_matches:
            all_matches = subset_matches
    # Prefer exact file from processed/shared-like directories if duplicates exist
    preferred = [
        p
        for p in all_matches
        if any(k in str(p).lower() for k in ["processed", "shared", "k-auxi", "nckh"])
    ]
    chosen = preferred[0] if preferred else all_matches[0]
    print(f"[DATA] {filename} (subset={subset}): {chosen}")
    return chosen


def norm_text(x):
    return re.sub(r"\s+", " ", str(x).lower()).strip()


def non_null_id_set(series):
    if series is None:
        return set()
    out = set()
    for x in pd.Series(series).dropna():
        s = str(x).strip()
        if re.fullmatch(r"-?\d+\.0", s):
            s = s[:-2]
        out.add(s)
    return out


# ============================================================
# FROZEN CORE FILES
# ============================================================
ORIGINAL_TRAIN_PATH = find_one("train.csv")
VALIDATION_PATH = find_one("val.csv")
TEST_PATH = find_one("test.csv")
MLB_PATH = find_one("multilabel_binarizer.pkl")

original_train_df = pd.read_csv(ORIGINAL_TRAIN_PATH)
val_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

for frame in [original_train_df, val_df, test_df]:
    frame["Cleaned_Text"] = frame["Cleaned_Text"].fillna("").astype(str)

# ============================================================
# RESOLVE STAGE-1 DATA FROM THE REGISTRY
# ============================================================
STAGE1_SOURCE_FILES = []

if EXP["stage1_mode"] == "original":
    train_df = original_train_df.copy()
    STAGE1_SOURCE_FILES = [ORIGINAL_TRAIN_PATH.name]

else:
    # Augmented mode (Cyber / EDA)
    AUG_PATH = find_one("train_augmented_generic_eda.csv")
    train_df = pd.read_csv(AUG_PATH)
    train_df["Cleaned_Text"] = train_df["Cleaned_Text"].fillna("").astype(str)
    STAGE1_SOURCE_FILES = [AUG_PATH.name]

stage2_train_df = original_train_df.copy() if EXP["stage2"] else None

# ============================================================
# LOAD LABEL SPACE
# ============================================================
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with open(MLB_PATH, "rb") as f:
        saved_mlb = pickle.load(f)

classes = np.asarray(saved_mlb.classes_, dtype=str)
mlb = MultiLabelBinarizer(classes=classes)
mlb.fit([[]])
NUM_LABELS = len(classes)
label_to_idx = {label: i for i, label in enumerate(classes)}


def parse_labels(value):
    return [x.strip() for x in str(value).split(",") if x.strip() and x.strip() != "nan"]


def encode_labels(series):
    return mlb.transform(series.map(parse_labels)).astype(np.float32)


y_original_train = encode_labels(original_train_df["Labels"])
y_train = encode_labels(train_df["Labels"])
y_stage2 = encode_labels(stage2_train_df["Labels"]) if stage2_train_df is not None else None
y_val = encode_labels(val_df["Labels"])
y_test = encode_labels(test_df["Labels"])

original_train_support = y_original_train.sum(axis=0)
stage1_train_support = y_train.sum(axis=0)

# ============================================================
# STRICT SHARED-DATA SANITY CHECKS
# ============================================================
print("\n" + "=" * 78)
print("SHARED DATA SANITY CHECKS")

orig_norm = set(original_train_df["Cleaned_Text"].map(norm_text))
val_norm = set(val_df["Cleaned_Text"].map(norm_text))
test_norm = set(test_df["Cleaned_Text"].map(norm_text))
stage1_norm = set(train_df["Cleaned_Text"].map(norm_text))

if orig_norm & val_norm:
    raise RuntimeError("Primary train overlaps Validation.")
if orig_norm & test_norm:
    raise RuntimeError("Primary train overlaps Test.")
if stage1_norm & val_norm:
    raise RuntimeError("Stage-1 train overlaps Validation.")
if stage1_norm & test_norm:
    raise RuntimeError("Stage-1 train overlaps Test.")

stage1_ids = non_null_id_set(train_df.get("source_sample_id"))
val_ids = non_null_id_set(val_df.get("source_sample_id"))
test_ids = non_null_id_set(test_df.get("source_sample_id"))

if stage1_ids and val_ids and (stage1_ids & val_ids):
    raise RuntimeError("Stage-1 source IDs overlap Validation.")
if stage1_ids and test_ids and (stage1_ids & test_ids):
    raise RuntimeError("Stage-1 source IDs overlap Test.")
if val_ids and test_ids and (val_ids & test_ids):
    raise RuntimeError("Validation source IDs overlap Test.")

if "is_augmented" in val_df.columns and val_df["is_augmented"].fillna(0).astype(int).sum() != 0:
    raise RuntimeError("Validation contains augmented rows.")
if "is_augmented" in test_df.columns and test_df["is_augmented"].fillna(0).astype(int).sum() != 0:
    raise RuntimeError("Test contains augmented rows.")

EXPECTED_BENCHMARKS = {
    "joint": {"train": 15016, "val": 2145, "test": 4291, "labels": 188},
    "cti_to_mitre": {"train": 9060, "val": 1295, "test": 2599, "labels": 188},
    "tram": {"train": 5955, "val": 851, "test": 1705, "labels": 50},
}
if DATASET_SUBSET in EXPECTED_BENCHMARKS:
    exp = EXPECTED_BENCHMARKS[DATASET_SUBSET]
    print(f"[INFO] Target Benchmark: '{DATASET_SUBSET}' (expected ~{exp['train']:,} train, ~{exp['val']:,} val, ~{exp['test']:,} test, {exp['labels']} labels)")

n_aug = 0
if "is_augmented" in train_df.columns:
    n_aug = int(train_df["is_augmented"].fillna(0).astype(int).eq(1).sum())

print(f"Scenario: {SCENARIO}")
print(f"Dataset subset: {DATASET_SUBSET}")
print(f"Primary rows: {len(original_train_df):,}")
print(f"Stage-1 rows: {len(train_df):,}")
print(f"Stage-1 augmented rows: {n_aug:,}")
print(f"Validation rows: {len(val_df):,}")
print(f"Test rows: {len(test_df):,}")
print(f"Labels: {NUM_LABELS}")
print(f"Stage-2 enabled: {EXP['stage2']}")
print(f"Minority-F2 enabled: {EXP['minority_f2']}")
print("[OK] Shared data checks passed.")
print("=" * 78)

## Dataset / loader helpers

In [ ]:
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed); random.seed(worker_seed)

class TextDataset(Dataset):
    def __init__(self, texts, labels, ids):
        self.texts = list(texts); self.labels = np.asarray(labels, np.float32); self.ids = np.asarray(ids)
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i], self.labels[i], self.ids[i]

def make_loader(texts, labels, ids, tokenizer, max_length, batch_size, shuffle, seed):
    ds = TextDataset(texts, labels, ids)
    def collate(batch):
        text, y, sample_id = zip(*batch)
        tok = tokenizer(list(text), padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        tok["labels"] = torch.tensor(np.stack(y), dtype=torch.float32)
        tok["sample_ids"] = np.asarray(sample_id)
        return tok
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate,
                      num_workers=CONFIG["num_workers"], pin_memory=True,
                      worker_init_fn=seed_worker, generator=g, persistent_workers=CONFIG["num_workers"]>0)

## Metrics and threshold tuning

In [ ]:
METRIC_KEYS = ["micro_precision","micro_recall","micro_f1","macro_precision","macro_recall",
               "macro_f1","weighted_f1","hamming_loss"]

def classification_metrics(y_true, y_pred):
    return {
        "micro_precision": precision_score(y_true,y_pred,average="micro",zero_division=0),
        "micro_recall": recall_score(y_true,y_pred,average="micro",zero_division=0),
        "micro_f1": f1_score(y_true,y_pred,average="micro",zero_division=0),
        "macro_precision": precision_score(y_true,y_pred,average="macro",zero_division=0),
        "macro_recall": recall_score(y_true,y_pred,average="macro",zero_division=0),
        "macro_f1": f1_score(y_true,y_pred,average="macro",zero_division=0),
        "weighted_f1": f1_score(y_true,y_pred,average="weighted",zero_division=0),
        "hamming_loss": hamming_loss(y_true,y_pred),
        "avg_predicted_labels": float(y_pred.sum(1).mean()),
    }

def ranking_metrics(y_true, scores, ks=(1,3,5,10,20,50)):
    order = np.argsort(-scores, axis=1)
    true_count = np.maximum(y_true.sum(1), 1)
    out = {}
    for k in ks:
        kk=min(k,scores.shape[1]); hits=np.take_along_axis(y_true,order[:,:kk],axis=1).sum(1)
        out[f"precision_at_{k}"] = float(np.mean(hits/kk))
        out[f"recall_at_{k}"] = float(np.mean(hits/true_count))
        out[f"hit_at_{k}"] = float(np.mean(hits>0))
    ranks=[]; aps=[]
    for i in range(len(y_true)):
        rel=y_true[i,order[i]].astype(bool); pos=np.flatnonzero(rel)
        ranks.append((pos[0]+1) if len(pos) else scores.shape[1]+1)
        if len(pos): aps.append(np.mean([(j+1)/(p+1) for j,p in enumerate(pos)]))
        else: aps.append(0.0)
    out["mrr"]=float(np.mean(1/np.asarray(ranks))); out["map"]=float(np.mean(aps))
    return out

def threshold_sweep(y_true, probs):
    thresholds=np.round(np.arange(CONFIG["threshold_min"], CONFIG["threshold_max"]+1e-9,
                                  CONFIG["threshold_step"]), 10)
    rows=[]
    for t in thresholds:
        m=classification_metrics(y_true,(probs>=t).astype(np.uint8)); m["threshold"]=float(t); rows.append(m)
    return pd.DataFrame(rows)

def tune_thresholds(y_true, probs):
    sweep=threshold_sweep(y_true,probs)
    best_micro=float(sweep.loc[sweep.micro_f1.idxmax(),"threshold"])
    best_macro=float(sweep.loc[sweep.macro_f1.idxmax(),"threshold"])
    support=y_true.sum(0).astype(int); per=np.full(y_true.shape[1],best_micro,dtype=np.float32); rows=[]
    for j in range(y_true.shape[1]):
        fallback=support[j] < CONFIG["min_val_support_per_label"]
        if not fallback:
            vals=[]
            for t in sweep.threshold:
                pred=(probs[:,j]>=t).astype(np.uint8)
                vals.append(f1_score(y_true[:,j],pred,zero_division=0))
            per[j]=float(sweep.threshold.iloc[int(np.argmax(vals))])
        pred=(probs[:,j]>=per[j]).astype(np.uint8)
        rows.append({"technique_id":classes[j],"validation_support":support[j],"optimal_threshold":float(per[j]),
                     "validation_precision":precision_score(y_true[:,j],pred,zero_division=0),
                     "validation_recall":recall_score(y_true[:,j],pred,zero_division=0),
                     "validation_f1":f1_score(y_true[:,j],pred,zero_division=0),"fallback_used":bool(fallback)})
    return best_micro,best_macro,per,sweep,pd.DataFrame(rows)



def minority_macro_fbeta(y_true, y_pred, minority_mask, beta=2.0):
    """
    Mean per-label F-beta over the fixed minority label set.
    Labels are defined from original train support, not Validation/Test.
    """
    idxs = np.where(minority_mask)[0]
    if len(idxs) == 0:
        return 0.0

    vals = [
        fbeta_score(
            y_true[:, j],
            y_pred[:, j],
            beta=beta,
            zero_division=0
        )
        for j in idxs
    ]
    return float(np.mean(vals))


def minority_macro_recall(y_true, y_pred, minority_mask):
    idxs = np.where(minority_mask)[0]
    if len(idxs) == 0:
        return 0.0

    vals = [
        recall_score(
            y_true[:, j],
            y_pred[:, j],
            zero_division=0
        )
        for j in idxs
    ]
    return float(np.mean(vals))


def tune_minority_f2_threshold(
    y_true,
    probs,
    original_train_support,
    global_threshold,
    support_threshold=20,
    beta=2.0
):
    """
    Hybrid thresholding:
      - non-minority labels: global threshold
      - minority labels: one shared threshold tuned on Validation Macro-Fbeta

    Tie-breaking:
      1) highest minority Macro-Fbeta
      2) highest minority Macro Recall
      3) highest overall Micro-F1
    """
    minority_mask = original_train_support < support_threshold

    thresholds = np.round(
        np.arange(
            CONFIG["threshold_min"],
            CONFIG["threshold_max"] + 1e-9,
            CONFIG["threshold_step"]
        ),
        10
    )

    rows = []

    for minority_t in thresholds:
        threshold_vector = np.full(
            probs.shape[1],
            global_threshold,
            dtype=np.float32
        )
        threshold_vector[minority_mask] = float(minority_t)

        pred = (
            probs >= threshold_vector[None, :]
        ).astype(np.uint8)

        overall = classification_metrics(y_true, pred)

        rows.append({
            "minority_threshold": float(minority_t),
            "global_threshold": float(global_threshold),
            "minority_macro_f2": minority_macro_fbeta(
                y_true,
                pred,
                minority_mask,
                beta=beta
            ),
            "minority_macro_recall": minority_macro_recall(
                y_true,
                pred,
                minority_mask
            ),
            "overall_micro_precision": overall["micro_precision"],
            "overall_micro_recall": overall["micro_recall"],
            "overall_micro_f1": overall["micro_f1"],
            "overall_macro_precision": overall["macro_precision"],
            "overall_macro_recall": overall["macro_recall"],
            "overall_macro_f1": overall["macro_f1"],
            "hamming_loss": overall["hamming_loss"]
        })

    sweep = pd.DataFrame(rows)

    best = (
        sweep.sort_values(
            by=[
                "minority_macro_f2",
                "minority_macro_recall",
                "overall_micro_f1"
            ],
            ascending=[False, False, False]
        )
        .iloc[0]
    )

    best_minority_t = float(
        best["minority_threshold"]
    )

    final_thresholds = np.full(
        probs.shape[1],
        global_threshold,
        dtype=np.float32
    )

    final_thresholds[
        minority_mask
    ] = best_minority_t

    return (
        best_minority_t,
        final_thresholds,
        minority_mask,
        sweep
    )

def label_metrics(y_true,y_pred,train_support,val_support,thresholds,groups):
    rows=[]
    for j,tid in enumerate(classes):
        rows.append({"Technique_ID":tid,"Train_Support":int(train_support[j]),"Validation_Support":int(val_support[j]),
                     "Test_Support":int(y_true[:,j].sum()),"Precision":precision_score(y_true[:,j],y_pred[:,j],zero_division=0),
                     "Recall":recall_score(y_true[:,j],y_pred[:,j],zero_division=0),
                     "F1":f1_score(y_true[:,j],y_pred[:,j],zero_division=0),
                     "Optimal_Threshold":float(thresholds[j]),"Frequency_Group":groups[j]})
    return pd.DataFrame(rows)

def frequency_groups(original_train_support):
    groups = []
    for x in original_train_support:
        if x >= 100: groups.append("Head")
        elif 30 <= x < 100: groups.append("Medium")
        else: groups.append("Tail")
    return np.asarray(groups), 30.0, 100.0

def print_experiment_summary(result):
    m=result["test_per_label"]
    names=[("Micro Precision","micro_precision"),("Micro Recall","micro_recall"),("Micro-F1","micro_f1"),
           ("Macro Precision","macro_precision"),("Macro Recall","macro_recall"),("Macro-F1","macro_f1"),
           ("Weighted-F1","weighted_f1"),("Hamming Loss","hamming_loss"),
           ("Precision@3","precision_at_3"),("Recall@3","recall_at_3"),("Hit@3","hit_at_3"),
           ("Precision@5","precision_at_5"),("Recall@5","recall_at_5"),("Hit@5","hit_at_5")]
    rows=[{"Metric":label,"Test value":m[key]} for label,key in names if key in m]
    print("\n"+"="*70)
    print(f"EXPERIMENT COMPLETE: {result['model']} | Seed {result['seed']}")
    print("="*70)
    print(f"Best epoch: {result['best_epoch']} | Selection: {result['selection_metric']} = {result['validation_score']:.4f}")
    print(f"Validation-selected global threshold: {result['global_threshold']:.2f}")
    display(pd.DataFrame(rows).set_index("Metric").round(4))
    if result.get("retrieval"):
        retrieval=pd.DataFrame([{"Metric":k.replace("_at_","@").replace("_"," ").title(),"Test value":v} for k,v in result["retrieval"].items()])
        print("Bi-Encoder ranking metrics:"); display(retrieval.set_index("Metric").round(4))
    print(f"Training: {result['training_seconds']/60:.2f} min | Inference: {result['inference_ms_per_sample']:.3f} ms/sample")
    print(f"Peak VRAM: {result['peak_vram_mb']:.1f} MB | Model size: {result['model_size_mb']:.1f} MB | Device: {result['device']}")
    print("="*70)

## Model and ASL

In [ ]:
def masked_mean(last_hidden, attention_mask):
    mask=attention_mask.unsqueeze(-1).to(last_hidden.dtype)
    return (last_hidden*mask).sum(1)/mask.sum(1).clamp_min(1e-9)

def load_bert_encoder(checkpoint):
    model_config=AutoConfig.from_pretrained(checkpoint)
    if hasattr(model_config,"reference_compile"): model_config.reference_compile=False
    return AutoModel.from_pretrained(checkpoint,config=model_config,attn_implementation="eager")

class BertClassifier(nn.Module):
    def __init__(self, checkpoint, num_labels):
        super().__init__(); self.encoder=load_bert_encoder(checkpoint)
        hidden=self.encoder.config.hidden_size
        self.dropout=nn.Dropout(getattr(self.encoder.config,"classifier_dropout",0.1) or 0.1)
        self.classifier=nn.Linear(hidden,num_labels)
    def forward(self,input_ids,attention_mask,**kwargs):
        out=self.encoder(input_ids=input_ids,attention_mask=attention_mask)
        return self.classifier(self.dropout(masked_mean(out.last_hidden_state,attention_mask)))

class AsymmetricLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,logits,targets):
        p=torch.sigmoid(logits); pos=p; neg=1-p
        if CONFIG["asl_clip"]:
            neg=(neg+CONFIG["asl_clip"]).clamp(max=1)
        loss=targets*torch.log(pos.clamp_min(CONFIG["asl_eps"]))+(1-targets)*torch.log(neg.clamp_min(CONFIG["asl_eps"]))
        pt=pos*targets+neg*(1-targets)
        weight=torch.pow((1-pt).clamp_min(0), CONFIG["gamma_pos"]*targets+CONFIG["gamma_neg"]*(1-targets))
        return -(loss*weight).sum(1).mean()

class Encoder(nn.Module):
    def __init__(self,checkpoint):
        super().__init__(); self.backbone=load_bert_encoder(checkpoint)
    def forward(self,input_ids,attention_mask,**kwargs):
        out=self.backbone(input_ids=input_ids,attention_mask=attention_mask)
        return F.normalize(masked_mean(out.last_hidden_state,attention_mask),p=2,dim=1)

# ASL sanity checks
_loss=AsymmetricLoss(); _z=torch.tensor([[5.,-5.],[-5.,5.]],requires_grad=True); _y=torch.tensor([[1.,0.],[0.,1.]])
_good=_loss(_z,_y); _bad=_loss(-_z,_y); _good.backward()
assert torch.isfinite(_good) and _good<_bad and _z.grad is not None and torch.isfinite(_z.grad).all()
print("[OK] ASL finite, direction and gradient checks passed")

## Optimizer / training helpers

In [ ]:
def optimizer_and_scheduler(model, loader_len, epochs, lr):
    no_decay=("bias","LayerNorm.weight","layer_norm.weight")
    params=[{"params":[p for n,p in model.named_parameters() if p.requires_grad and not any(x in n for x in no_decay)],"weight_decay":CONFIG["weight_decay"]},
            {"params":[p for n,p in model.named_parameters() if p.requires_grad and any(x in n for x in no_decay)],"weight_decay":0.0}]
    opt=torch.optim.AdamW(params,lr=lr)
    steps=math.ceil(loader_len/CONFIG["gradient_accumulation_steps"])*epochs
    sch=get_linear_schedule_with_warmup(opt,int(steps*CONFIG["warmup_ratio"]),steps)
    return opt,sch

def to_device(batch):
    ids=batch.pop("sample_ids"); y=batch.pop("labels").to(DEVICE,non_blocking=True)
    x={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}
    return x,y,ids

@torch.no_grad()
def predict_classifier(model,loader,desc="Evaluating SecureBERT"):
    model.eval(); ys=[]; probs=[]; ids=[]
    progress=tqdm(loader,desc=desc,leave=False,dynamic_ncols=True)
    for batch in progress:
        x,y,sid=to_device(batch); logits=model(**x)
        ys.append(y.cpu().numpy()); probs.append(torch.sigmoid(logits).cpu().numpy()); ids.extend(sid.tolist())
    return np.concatenate(ys),np.concatenate(probs),np.asarray(ids)

def train_classifier_epoch(model,loader,loss_fn,opt,sch,scaler,desc):
    model.train()
    opt.zero_grad(set_to_none=True)
    total = 0.0
    amp_skipped = 0
    optimizer_attempts = 0

    progress = tqdm(
        enumerate(loader, 1),
        total=len(loader),
        desc=desc,
        leave=True,
        dynamic_ncols=True
    )

    for step, batch in progress:
        x, y, _ = to_device(batch)

        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(**x), y) / CONFIG["gradient_accumulation_steps"]

        if not torch.isfinite(loss):
            raise FloatingPointError("NaN/Inf ASL loss")

        scaler.scale(loss).backward()
        batch_loss = loss.item() * CONFIG["gradient_accumulation_steps"]
        total += batch_loss

        if step % CONFIG["gradient_accumulation_steps"] == 0 or step == len(loader):
            optimizer_attempts += 1
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])

            old_scale = scaler.get_scale()
            scaler.step(opt)
            scaler.update()

            if scaler.get_scale() >= old_scale:
                sch.step()
            else:
                amp_skipped += 1
                print(
                    f"[AMP] Optimizer step skipped after gradient overflow at batch {step}; "
                    "scheduler unchanged.",
                    flush=True
                )

            opt.zero_grad(set_to_none=True)

        progress.set_postfix(
            loss=f"{batch_loss:.4f}",
            avg=f"{total/step:.4f}",
            lr=f"{opt.param_groups[0]['lr']:.2e}",
            vram=f"{torch.cuda.memory_allocated()/1024**3:.1f}GB"
        )

        if step % CONFIG["log_every_batches"] == 0 or step == len(loader):
            print(
                f"[PROGRESS] {desc} | batch {step}/{len(loader)} | "
                f"loss={batch_loss:.4f} | avg={total/step:.4f} | "
                f"lr={opt.param_groups[0]['lr']:.2e} | "
                f"VRAM={torch.cuda.memory_allocated()/1024**3:.1f}GB",
                flush=True
            )

    avg_loss = total / max(1, len(loader))
    return avg_loss, amp_skipped, optimizer_attempts

## Locked protocol

- BERT: `google-bert/bert-base-uncased`
- Controlled-ablation seed: `42`
- Max length: `384`
- ASL: `gamma_pos=1`, `gamma_neg=4`, `clip=0.05`
- LR: `2e-5`
- Epochs: Stage 1 = 6; Stage 2 = 6
- Checkpoint selection: Validation Macro-F1 @ 0.5
- Global threshold: selected on Validation only
- Minority: original-train support `<20`
- Minority threshold objective: Macro-F2, beta=2
- Test is evaluated after Validation selection; never used to select configuration


In [ ]:

def experiment_run_dir(seed):
    return RESULTS / f"seed_{seed}"


def fit_stage(
    model,
    train_loader,
    val_loader,
    loss_fn,
    seed,
    stage_name,
    epochs,
    checkpoint,
    f_log
):
    opt, sch = optimizer_and_scheduler(
        model,
        len(train_loader),
        epochs,
        CONFIG["learning_rate"]
    )
    scaler = torch.cuda.amp.GradScaler()

    history = []
    best_macro_f1 = -1.0
    best_epoch = 0
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        loss, amp_skipped, amp_attempts = train_classifier_epoch(
            model,
            train_loader,
            loss_fn,
            opt,
            sch,
            scaler,
            desc=f"{stage_name} | seed {seed} | epoch {epoch}/{epochs}"
        )
        epoch_seconds = time.time() - t0

        vy, vp, _ = predict_classifier(
            model,
            val_loader,
            desc=f"{stage_name} validation | seed {seed} | epoch {epoch}"
        )
        vm = classification_metrics(vy, (vp >= 0.5).astype(np.uint8))
        current = vm["macro_f1"]

        history.append({
            "stage": stage_name,
            "epoch": epoch,
            "train_loss": loss,
            "val_micro_precision": vm["micro_precision"],
            "val_micro_recall": vm["micro_recall"],
            "val_micro_f1": vm["micro_f1"],
            "val_macro_precision": vm["macro_precision"],
            "val_macro_recall": vm["macro_recall"],
            "val_macro_f1": vm["macro_f1"],
            "val_weighted_f1": vm["weighted_f1"],
            "val_hamming_loss": vm["hamming_loss"],
            "learning_rate": opt.param_groups[0]["lr"],
            "epoch_seconds": epoch_seconds,
            "amp_skipped_steps": amp_skipped,
            "amp_skip_rate": 100.0 * amp_skipped / max(1, amp_attempts),
        })

        if current > best_macro_f1:
            best_macro_f1 = current
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            patience_counter += 1

        msg = (
            "\n" + "=" * 70 + "\n"
            f"{stage_name} — EPOCH {epoch}/{epochs}\n"
            + "=" * 70 + "\n"
            f"Train Loss: {loss:.6f}\n"
            f"Validation Micro F1: {vm['micro_f1']*100:.2f}%\n"
            f"Validation Macro F1: {vm['macro_f1']*100:.2f}%\n"
            f"Validation Macro Recall: {vm['macro_recall']*100:.2f}%\n"
            f"Best Epoch: {best_epoch}\n"
            f"Best Validation Macro F1 @0.5: {best_macro_f1*100:.2f}%\n"
            f"Patience: {patience_counter}/{CONFIG['early_stopping_patience']}\n"
            + "=" * 70 + "\n"
        )
        print(msg)
        f_log.write(msg)
        f_log.flush()

        if patience_counter >= CONFIG["early_stopping_patience"]:
            print(f"[EARLY STOP] {stage_name} at epoch {epoch}; best={best_epoch}")
            break

    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE))

    del opt, sch, scaler
    gc.collect()
    torch.cuda.empty_cache()

    return model, pd.DataFrame(history), best_epoch, best_macro_f1


def fixed_bucket(support):
    support = int(support)
    if support < 20:
        return "Tail"
    if support < 100:
        return "Medium"
    return "Head"


def build_per_label(y_true, pred, thresholds, prefix):
    rows = []
    for j, label in enumerate(classes):
        rows.append({
            "Technique_ID": label,
            "Original_Train_Support": int(original_train_support[j]),
            "Stage1_Train_Support": int(stage1_train_support[j]),
            "Test_Support": int(y_true[:, j].sum()),
            "Precision": precision_score(y_true[:, j], pred[:, j], zero_division=0),
            "Recall": recall_score(y_true[:, j], pred[:, j], zero_division=0),
            "F1": f1_score(y_true[:, j], pred[:, j], zero_division=0),
            "TP": int(((y_true[:, j] == 1) & (pred[:, j] == 1)).sum()),
            "FP": int(((y_true[:, j] == 0) & (pred[:, j] == 1)).sum()),
            "FN": int(((y_true[:, j] == 1) & (pred[:, j] == 0)).sum()),
            "Threshold": float(thresholds[j]) if np.ndim(thresholds) else float(thresholds),
            "Support_Bucket": fixed_bucket(original_train_support[j]),
            "Prediction_Mode": prefix,
        })
    return pd.DataFrame(rows)


def run_bert(seed):
    run_dir = experiment_run_dir(seed)
    run_dir.mkdir(parents=True, exist_ok=True)

    final_marker = run_dir / "metrics.json"
    if final_marker.exists() and not CONFIG["force_rerun"]:
        print(f"[SKIP] {SCENARIO} seed={seed} already completed.")
        return

    print("\n" + "#" * 78)
    print(f"SCENARIO {SCENARIO} | {EXP['title']} | SEED {seed}")
    print("#" * 78)

    set_seed(seed)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    started = time.time()

    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_checkpoint"])

    stage1_loader = make_loader(
        train_df.Cleaned_Text,
        y_train,
        np.arange(len(train_df)),
        tokenizer,
        CONFIG["max_length"],
        CONFIG["train_batch_size"],
        True,
        seed
    )

    val_loader = make_loader(
        val_df.Cleaned_Text,
        y_val,
        np.arange(len(val_df)),
        tokenizer,
        CONFIG["max_length"],
        CONFIG["eval_batch_size"],
        False,
        seed
    )

    test_loader = make_loader(
        test_df.Cleaned_Text,
        y_test,
        np.arange(len(test_df)),
        tokenizer,
        CONFIG["max_length"],
        CONFIG["eval_batch_size"],
        False,
        seed
    )

    stage2_loader = None
    if EXP["stage2"]:
        stage2_loader = make_loader(
            stage2_train_df.Cleaned_Text,
            y_stage2,
            np.arange(len(stage2_train_df)),
            tokenizer,
            CONFIG["max_length"],
            CONFIG["train_batch_size"],
            True,
            seed
        )

    model = BertClassifier(
        CONFIG["model_checkpoint"],
        NUM_LABELS
    ).to(DEVICE)

    loss_fn = AsymmetricLoss()

    stage1_checkpoint = run_dir / "stage1_best_model.pt"
    stage2_checkpoint = run_dir / "stage2_best_model.pt"

    with open(run_dir / "training.log", "w") as f_log:
        f_log.write(
            f"Scenario={SCENARIO}\n"
            f"Seed={seed}\n"
            f"Stage1Mode={EXP['stage1_mode']}\n"
            f"Stage1Files={STAGE1_SOURCE_FILES}\n"
            f"Stage2={EXP['stage2']}\n"
            f"MinorityF2={EXP['minority_f2']}\n"
        )

        model, stage1_history, stage1_best_epoch, stage1_best_macro = fit_stage(
            model=model,
            train_loader=stage1_loader,
            val_loader=val_loader,
            loss_fn=loss_fn,
            seed=seed,
            stage_name=f"STAGE 1 — {EXP['stage1_mode'].upper()}",
            epochs=CONFIG["model_epochs"],
            checkpoint=stage1_checkpoint,
            f_log=f_log
        )
        stage1_history.to_csv(run_dir / "stage1_epoch_history.csv", index=False)

        if EXP["stage2"]:
            model, stage2_history, stage2_best_epoch, stage2_best_macro = fit_stage(
                model=model,
                train_loader=stage2_loader,
                val_loader=val_loader,
                loss_fn=loss_fn,
                seed=seed,
                stage_name="STAGE 2 — ORIGINAL ONLY",
                epochs=CONFIG["stage2_epochs"],
                checkpoint=stage2_checkpoint,
                f_log=f_log
            )
            stage2_history.to_csv(run_dir / "stage2_epoch_history.csv", index=False)
            final_checkpoint = stage2_checkpoint
        else:
            stage2_best_epoch = None
            stage2_best_macro = None
            final_checkpoint = stage1_checkpoint

    model.load_state_dict(torch.load(final_checkpoint, map_location=DEVICE))

    # ========================================================
    # VALIDATION-ONLY THRESHOLD SELECTION
    # ========================================================
    vy, vp, _ = predict_classifier(
        model,
        val_loader,
        desc=f"Validation threshold selection | {SCENARIO} | seed {seed}"
    )

    global_t, macro_t, per_t, sweep, threshold_table = tune_thresholds(vy, vp)
    sweep.to_csv(run_dir / "threshold_sweep.csv", index=False)

    pred_val_global = (vp >= global_t).astype(np.uint8)
    val_global = classification_metrics(vy, pred_val_global)

    minority_t = None
    minority_thresholds = None
    minority_mask = original_train_support < CONFIG["minority_support_threshold"]
    val_minority_metrics = None
    val_minority_f2 = None
    val_minority_recall = None

    if EXP["minority_f2"]:
        minority_t, minority_thresholds, minority_mask, minority_sweep = tune_minority_f2_threshold(
            y_true=vy,
            probs=vp,
            original_train_support=original_train_support,
            global_threshold=global_t,
            support_threshold=CONFIG["minority_support_threshold"],
            beta=CONFIG["minority_fbeta_beta"]
        )
        minority_sweep.to_csv(run_dir / "minority_f2_threshold_sweep.csv", index=False)

        pred_val_minority = (vp >= minority_thresholds[None, :]).astype(np.uint8)
        val_minority_metrics = classification_metrics(vy, pred_val_minority)
        val_minority_f2 = minority_macro_fbeta(
            vy, pred_val_minority, minority_mask, beta=CONFIG["minority_fbeta_beta"]
        )
        val_minority_recall = minority_macro_recall(
            vy, pred_val_minority, minority_mask
        )

    validation_selection = {
        "scenario": SCENARIO,
        "scenario_title": EXP["title"],
        "seed": seed,
        "stage1_mode": EXP["stage1_mode"],
        "stage1_source_files": STAGE1_SOURCE_FILES,
        "stage2": EXP["stage2"],
        "minority_f2": EXP["minority_f2"],
        "stage1_best_epoch": stage1_best_epoch,
        "stage1_best_macro_f1_at_05": stage1_best_macro,
        "stage2_best_epoch": stage2_best_epoch,
        "stage2_best_macro_f1_at_05": stage2_best_macro,
        "global_threshold": float(global_t),
        "macro_optimal_threshold": float(macro_t),
        "minority_threshold": float(minority_t) if minority_t is not None else None,
        "validation_global": val_global,
        "validation_minority_f2": val_minority_metrics,
        "validation_minority_macro_f2": float(val_minority_f2) if val_minority_f2 is not None else None,
        "validation_minority_macro_recall": float(val_minority_recall) if val_minority_recall is not None else None,
    }
    (run_dir / "validation_selection.json").write_text(
        json.dumps(validation_selection, indent=2)
    )

    # ========================================================
    # LOCKED TEST — NO TEST-BASED SELECTION
    # ========================================================
    ty, tp, tids = predict_classifier(
        model,
        test_loader,
        desc=f"LOCKED TEST | {SCENARIO} | seed {seed}"
    )

    pred_global = (tp >= global_t).astype(np.uint8)
    pred_per = (tp >= per_t[None, :]).astype(np.uint8)

    global_metrics = classification_metrics(ty, pred_global)
    global_metrics.update(ranking_metrics(ty, tp, ks=(3, 5)))

    per_metrics = classification_metrics(ty, pred_per)
    per_metrics.update(ranking_metrics(ty, tp, ks=(3, 5)))

    test_minority_metrics = None
    test_minority_f2 = None
    test_minority_recall = None
    pred_minority = None

    if EXP["minority_f2"]:
        pred_minority = (tp >= minority_thresholds[None, :]).astype(np.uint8)
        test_minority_metrics = classification_metrics(ty, pred_minority)
        test_minority_metrics.update(ranking_metrics(ty, tp, ks=(3, 5)))
        test_minority_f2 = minority_macro_fbeta(
            ty, pred_minority, minority_mask, beta=CONFIG["minority_fbeta_beta"]
        )
        test_minority_recall = minority_macro_recall(
            ty, pred_minority, minority_mask
        )

    # ========================================================
    # CONSISTENT OUTPUTS FOR EVERY TEAM MEMBER
    # ========================================================
    global_pl = build_per_label(ty, pred_global, global_t, "global")
    global_pl.to_csv(run_dir / "per_label_global.csv", index=False)

    if EXP["minority_f2"]:
        minority_pl = build_per_label(
            ty, pred_minority, minority_thresholds, "minority_f2"
        )
        minority_pl.to_csv(run_dir / "per_label_minority_f2.csv", index=False)

    save_kwargs = dict(
        sample_ids=tids,
        y_true=ty,
        probabilities=tp,
        predictions_global=pred_global,
        predictions_per_label=pred_per,
        global_threshold=np.asarray([global_t], dtype=np.float32),
        per_label_thresholds=per_t,
    )
    if EXP["minority_f2"]:
        save_kwargs["predictions_minority_f2"] = pred_minority
        save_kwargs["minority_thresholds"] = minority_thresholds

    np.savez_compressed(run_dir / "predictions.npz", **save_kwargs)

    resolved_config = {
        "scenario": SCENARIO,
        "scenario_title": EXP["title"],
        "seed": seed,
        "experiment_registry": EXP,
        "training_config": CONFIG,
        "stage1_source_files": STAGE1_SOURCE_FILES,
        "primary_train_rows": len(original_train_df),
        "stage1_rows": len(train_df),
        "validation_rows": len(val_df),
        "test_rows": len(test_df),
        "num_labels": NUM_LABELS,
    }
    (run_dir / "config_resolved.json").write_text(
        json.dumps(resolved_config, indent=2)
    )

    result = {
        "scenario": SCENARIO,
        "scenario_title": EXP["title"],
        "seed": seed,
        "stage1_best_epoch": stage1_best_epoch,
        "stage1_best_macro_f1_at_05": stage1_best_macro,
        "stage2_best_epoch": stage2_best_epoch,
        "stage2_best_macro_f1_at_05": stage2_best_macro,
        "selected_global_threshold": float(global_t),
        "selected_minority_threshold": float(minority_t) if minority_t is not None else None,
        "test_global": global_metrics,
        "test_per_label": per_metrics,
        "test_minority_f2": test_minority_metrics,
        "test_minority_macro_f2": float(test_minority_f2) if test_minority_f2 is not None else None,
        "test_minority_macro_recall": float(test_minority_recall) if test_minority_recall is not None else None,
        "training_seconds": time.time() - started,
        "peak_vram_mb": torch.cuda.max_memory_allocated() / 1024**2,
    }
    final_marker.write_text(json.dumps(result, indent=2))

    print("\n" + "=" * 78)
    print(f"FINAL TEST — {SCENARIO} — GLOBAL")
    print("=" * 78)
    print(f"Global threshold: {global_t:.2f}")
    print(f"Micro P/R/F1: {global_metrics['micro_precision']*100:.2f} / {global_metrics['micro_recall']*100:.2f} / {global_metrics['micro_f1']*100:.2f}")
    print(f"Macro P/R/F1: {global_metrics['macro_precision']*100:.2f} / {global_metrics['macro_recall']*100:.2f} / {global_metrics['macro_f1']*100:.2f}")
    print(f"Weighted F1: {global_metrics['weighted_f1']*100:.2f}")
    print(f"Hit@3 / Hit@5: {global_metrics.get('hit_at_3', 0)*100:.2f} / {global_metrics.get('hit_at_5', 0)*100:.2f}")
    print(f"MRR / MAP: {global_metrics.get('mrr', 0):.4f} / {global_metrics.get('map', 0):.4f}")

    if EXP["minority_f2"]:
        print("\n" + "=" * 78)
        label = "E1" if SCENARIO == "B2_E1" else f"{SCENARIO}-MINORITY-F2"
        print(f"FINAL TEST — {label}")
        print("=" * 78)
        print(f"Minority threshold: {minority_t:.2f}")
        print(f"Micro P/R/F1: {test_minority_metrics['micro_precision']*100:.2f} / {test_minority_metrics['micro_recall']*100:.2f} / {test_minority_metrics['micro_f1']*100:.2f}")
        print(f"Macro P/R/F1: {test_minority_metrics['macro_precision']*100:.2f} / {test_minority_metrics['macro_recall']*100:.2f} / {test_minority_metrics['macro_f1']*100:.2f}")
        print(f"Minority Macro-F2: {test_minority_f2*100:.2f}")
        print(f"Minority Macro Recall: {test_minority_recall*100:.2f}")

    del model, stage1_loader, val_loader, test_loader
    if stage2_loader is not None:
        del stage2_loader
    gc.collect()
    torch.cuda.empty_cache()


run_bert(CONFIG["seed"])
